# Bulk Feature Extraction
一次性批量提取 eGeMAPS / XLSR 特征，避免在每个训练 Notebook 中重复耗时的特征准备。

In [1]:
import sys
from pathlib import Path
import torch

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import PROJECT_ROOT, RANDOM_SEED
from data_split import create_train_val_split
from extract_egemap_feature import extract_egemaps_features_from_csv
from extract_XLSR_feature import extract_features_from_csv
from model import SSLModel

DEVICE = ("cuda" if torch.cuda.is_available() else
          ("mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else "cpu"))
FREEZE_XLSR = True  # 提取阶段通常冻结 XLSR 参数
TRAIN_RATIO = 0.8
SPLIT_RANDOM_SEED = RANDOM_SEED


/opt/anaconda3/envs/madress-2023-x86/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASETS = {
    "Pitt": {
        "raw_audio_dir": PROJECT_ROOT / "data/raw/Pitt",
        "egemap_csv": [
            PROJECT_ROOT / "data/processed/Pitt-egemap-train.csv",
            PROJECT_ROOT / "data/processed/Pitt-egemap-val.csv",
        ],
        "egemap_feature_dir": PROJECT_ROOT / "data/processed/Pitt_egemap_features",
        "xlsr_csv": [
            PROJECT_ROOT / "data/processed/Pitt-xlsr-train.csv",
            PROJECT_ROOT / "data/processed/Pitt-xlsr-val.csv",
        ],
        "xlsr_feature_dir": PROJECT_ROOT / "data/processed/Pitt_xlsr_features",
    },
    "ADReSS": {
        "raw_audio_dir": PROJECT_ROOT / "data/raw/ADReSS",
        "egemap_csv": [
            PROJECT_ROOT / "data/processed/ADReSS-egemap-train.csv",
            PROJECT_ROOT / "data/processed/ADReSS-egemap-val.csv",
        ],
        "egemap_feature_dir": PROJECT_ROOT / "data/processed/ADReSS_egemap_features",
        "xlsr_csv": [
            PROJECT_ROOT / "data/processed/ADReSS-xlsr-train.csv",
            PROJECT_ROOT / "data/processed/ADReSS-xlsr-val.csv",
        ],
        "xlsr_feature_dir": PROJECT_ROOT / "data/processed/ADReSS_xlsr_features",
    },
    "Lu": {
        "raw_audio_dir": PROJECT_ROOT / "data/raw/Lu",
        "egemap_csv": [
            PROJECT_ROOT / "data/processed/Lu-egemap-train.csv",
            PROJECT_ROOT / "data/processed/Lu-egemap-val.csv",
        ],
        "egemap_feature_dir": PROJECT_ROOT / "data/processed/Lu_egemap_features",
        "xlsr_csv": [
            PROJECT_ROOT / "data/processed/Lu-xlsr-train.csv",
            PROJECT_ROOT / "data/processed/Lu-xlsr-val.csv",
        ],
        "xlsr_feature_dir": PROJECT_ROOT / "data/processed/Lu_xlsr_features",
    }
    # "ADReSSo": {
    #     "raw_audio_dir": PROJECT_ROOT / "data/raw/ADReSSo",
    #     "egemap_csv": [
    #         PROJECT_ROOT / "data/processed/ADReSSo-egemap-train.csv",
    #         PROJECT_ROOT / "data/processed/ADReSSo-egemap-val.csv",
    #     ],
    #     "egemap_feature_dir": PROJECT_ROOT / "data/processed/ADReSSo_egemap_features",
    #     "xlsr_csv": [
    #         PROJECT_ROOT / "data/processed/ADReSSo-xlsr-train.csv",
    #         PROJECT_ROOT / "data/processed/ADReSSo-xlsr-val.csv",
    #     ],
    #     "xlsr_feature_dir": PROJECT_ROOT / "data/processed/ADReSSo_xlsr_features",
    # },
    # "ADReSS-M": {
    #     "raw_audio_dir": PROJECT_ROOT / "data/raw/ADReSS-M",
    #     "egemap_csv": [
    #         PROJECT_ROOT / "data/processed/ADReSS-M-egemap-train.csv",
    #         PROJECT_ROOT / "data/processed/ADReSS-M-egemap-val.csv",
    #     ],
    #     "egemap_feature_dir": PROJECT_ROOT / "data/processed/ADReSS-M_egemap_features",
    #     "xlsr_csv": [
    #         PROJECT_ROOT / "data/processed/ADReSS-M-xlsr-train.csv",
    #         PROJECT_ROOT / "data/processed/ADReSS-M-xlsr-val.csv",
    #     ],
    #     "xlsr_feature_dir": PROJECT_ROOT / "data/processed/ADReSS-M_xlsr_features",
    # },
}

SELECTED_DATASETS = ["Pitt", "ADReSS", "Lu"]#, "ADReSSo", "ADReSS-M"]
RUN_EGEMAP = True
RUN_XLSR = True


In [3]:
def ensure_csv_pair(dataset_name, csv_list, feature_dir, raw_audio_dir, *, is_xlsr):
    if not csv_list:
        return []
    if len(csv_list) != 2:
        raise ValueError(f"[{dataset_name}] 预期 train/val 两个 CSV，得到: {len(csv_list)}")
    train_csv, val_csv = csv_list
    if train_csv.exists() and val_csv.exists():
        return [train_csv, val_csv]
    if feature_dir is None:
        raise ValueError(f"[{dataset_name}] 未提供 feature_dir，无法自动生成 CSV")
    print(f"[{dataset_name}] ⚙️ 生成 {'XLSR' if is_xlsr else 'eGeMAP'} train/val CSV ...")
    create_train_val_split(
        raw_audio_dir=raw_audio_dir,
        train_csv_path=train_csv,
        val_csv_path=val_csv,
        feature_dir_name=feature_dir,
        train_ratio=TRAIN_RATIO,
        random_seed=SPLIT_RANDOM_SEED,
        dataset_name=dataset_name,
        xlsr=is_xlsr,
    )
    return [train_csv, val_csv]


def run_egemap_extraction(dataset_name, cfg):
    if not RUN_EGEMAP:
        return
    csv_list = ensure_csv_pair(
        dataset_name,
        cfg.get("egemap_csv", []),
        cfg.get("egemap_feature_dir"),
        cfg["raw_audio_dir"],
        is_xlsr=False,
    )
    for csv_path in csv_list:
        if not csv_path.exists():
            print(f"[{dataset_name}] ❌ eGeMAP CSV 不存在: {csv_path}")
            continue
        print(f"[{dataset_name}] ▶️ eGeMAP {csv_path.stem}")
        extract_egemaps_features_from_csv(csv_path, cfg["raw_audio_dir"])


def run_xlsr_extraction(dataset_name, cfg):
    if not RUN_XLSR:
        return
    csv_list = ensure_csv_pair(
        dataset_name,
        cfg.get("xlsr_csv", []),
        cfg.get("xlsr_feature_dir"),
        cfg["raw_audio_dir"],
        is_xlsr=True,
    )
    if not csv_list:
        return
    feature_dir = cfg.get("xlsr_feature_dir")
    if feature_dir is None:
        print(f"[{dataset_name}] ⚠️ 未配置 XLSR 特征目录，跳过")
        return
    feature_dir.mkdir(parents=True, exist_ok=True)
    ssl_model = SSLModel(device=DEVICE, freeze_xlsr=FREEZE_XLSR)
    for csv_path in csv_list:
        if not csv_path.exists():
            print(f"[{dataset_name}] ❌ XLSR CSV 不存在: {csv_path}")
            continue
        print(f"[{dataset_name}] ▶️ XLSR {csv_path.stem}")
        extract_features_from_csv(
            csv_path=csv_path,
            split_name=f"{dataset_name}-{csv_path.stem}",
            raw_audio_dir=cfg["raw_audio_dir"],
            xlsr_features_dir=feature_dir,
            device=DEVICE,
            ssl_model=ssl_model,
            freeze_xlsr=FREEZE_XLSR,
        )


for dataset_name in SELECTED_DATASETS:
    cfg = DATASETS.get(dataset_name)
    if cfg is None:
        print(f"[{dataset_name}] ⚠️ 未找到配置，跳过")
        continue
    print(f"\n==================== {dataset_name} ====================")
    run_egemap_extraction(dataset_name, cfg)
    run_xlsr_extraction(dataset_name, cfg)



==================== Pitt ====================
[Pitt] ▶️ eGeMAP Pitt-egemap-train
440 Audio Files


Extracting: 100%|██████████| 440/440 [00:00<00:00, 54047.14it/s]



============= Extraction completed! =============
Successfully extracted: 0 个
Skipped (Already Exists): 440 个
Total: 440 个
Errors: 0 个
[Pitt] ▶️ eGeMAP Pitt-egemap-val
111 Audio Files


Extracting: 100%|██████████| 111/111 [00:00<00:00, 41051.74it/s]


============= Extraction completed! =============
Successfully extracted: 0 个
Skipped (Already Exists): 111 个
Total: 111 个
Errors: 0 个
[Pitt] ⚙️ 生成 XLSR train/val CSV ...
============= Pitt Train/Val Split Complete! =============
Training set: 440 samples (Control: 193, Dementia: 247)
Validation set: 111 samples (Control: 49, Dementia: 62)

Training CSV path: /Users/sleepwalker/Library/Mobile Documents/com~apple~CloudDocs/Code-In-iCloud/Adaptation_is_all_you_need/ad_detection/data/processed/Pitt-xlsr-train.csv
Validation CSV path: /Users/sleepwalker/Library/Mobile Documents/com~apple~CloudDocs/Code-In-iCloud/Adaptation_is_all_you_need/ad_detection/data/processed/Pitt-xlsr-val.csv
XLSR:Using original XLSR model


...


============= Extraction completed! =============
Successfully extracted: 48 个
Skipped (Already Exists): 0 个
Total: 48 个
Errors: 0 个
[ADReSS-M] ⚙️ 生成 XLSR train/val CSV ...
============= ADReSS-M Train/Val Split Complete! =============
Training set: 189 samples (Control: 92, Dementia: 97)
Validation set: 48 samples (Control: 23, Dementia: 25)

Training CSV path: /Users/sleepwalker/Library/Mobile Documents/com~apple~CloudDocs/Code-In-iCloud/Adaptation_is_all_you_need/ad_detection/data/processed/ADReSS-M-xlsr-train.csv
Validation CSV path: /Users/sleepwalker/Library/Mobile Documents/com~apple~CloudDocs/Code-In-iCloud/Adaptation_is_all_you_need/ad_detection/data/processed/ADReSS-M-xlsr-val.csv
XLSR:Using original XLSR model
[ADReSS-M] ▶️ XLSR ADReSS-M-xlsr-train

============= Extracting XLSR features for ADReSS-M-ADReSS-M-xlsr-train =============
189 Audio Files


Extracting ADReSS-M-ADReSS-M-xlsr-train: 100%|██████████| 189/189 [20:30<00:00,  6.51s/it]


Successfully extracted: 189
Already exists (skipped): 0
Errors: 0
Total: 189
[ADReSS-M] ▶️ XLSR ADReSS-M-xlsr-val

============= Extracting XLSR features for ADReSS-M-ADReSS-M-xlsr-val =============
48 Audio Files


Extracting ADReSS-M-ADReSS-M-xlsr-val: 100%|██████████| 48/48 [09:39<00:00, 12.08s/it]

Successfully extracted: 48
Already exists (skipped): 0
Errors: 0
Total: 48
